In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# A digital twin of the SO-101, from first principles

A digital twin is only useful if code cannot tell it from the robot. That
reduces to three claims, each checked below with a measurement:

1. **It moves like the arm** - the same ros2_control controllers and MoveIt, within the same joint speed limits.
2. **It sees like the arm** - a camera rendered where the real one is mounted.
3. **It feels like the arm** - the jaws stall on contact, and the gripper action reports it.

With those in place, follow one command from words to a grasp: find the
named object with the wrist camera, and let MoveIt plan the pick.

Then the payoff: the twin console of `./run_pipeline.sh --digital-twin`, with
voice control - you speak, and the workshop's own behaviors drive the twin.

What is simulated: the arm, its servos, the wrist camera and the table
scene (MuJoCo). What is real: everything above the arm transport - your
webcam and microphone, YOLOv26s pose and detection, MediaPipe Hands, the
mapper, Whisper, the Llama intent parser and the behaviors. The twin is a
ROS 2 program of its own, the way the full system runs it: section 0 starts
it on a ROS domain of its own, and this kernel drives it with `MoveItArm`,
the same arm client the pipeline uses for the real arm. The notebook needs
ROS 2 from the first cell on.

## Compute placement

The pipeline is powered by AI models that utilize every subsystem of the APU. Each workload runs on the subsystem whose strengths best match its requirements.

<table style="border-collapse: collapse">
<thead>
<tr><th style="background-color: rgba(128, 128, 128, 0.2); border: 1px solid rgba(128, 128, 128, 0.4)">Task</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">Model or algorithm</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">What it does</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">Runs on</th><th style="border: 1px solid rgba(128, 128, 128, 0.4)">Runtime</th></tr>
</thead>
<tbody>
<tr style="background-color: rgba(0, 0, 0, 0)"><td rowspan="3" valign="top" style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">Voice control</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">WebRTC VAD, a Gaussian mixture model</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">finds where a spoken sentence starts and ends</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)"><code>webrtcvad</code> (C library)</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Whisper-base</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">speech → text</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">ONNX Runtime</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Llama 3.2 3B Instruct (Q4_K_M)</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">text → command</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">iGPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)"><code>llama.cpp</code> <code>llama-server</code> (HIP)</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td rowspan="2" valign="top" style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">Gestures</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">YOLOv26s-pose</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">finds your body and wrist</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">NPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">ONNX Runtime, VitisAI EP</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MediaPipe Hands</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">detects the hand gesture (pinch, roll)</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MediaPipe Tasks</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td rowspan="2" valign="top" style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">Object localization</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">YOLOv26s-detect</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">finds the named object in the arm camera</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">NPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">ONNX Runtime, VitisAI EP</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td style="border: 1px solid rgba(128, 128, 128, 0.4)">Localization and motion planning</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">places the object on the table and plans the grasp</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">NumPy, MoveIt Task Constructor</td></tr>
<tr style="background-color: rgba(0, 0, 0, 0)"><td valign="top" style="background-color: rgba(128, 128, 128, 0.2); vertical-align: top; border: 1px solid rgba(128, 128, 128, 0.4)">Digital twin</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MuJoCo 3D simulation</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">simulates the arm and the table scene</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">CPU (rendering on the iGPU)</td><td style="border: 1px solid rgba(128, 128, 128, 0.4)">MuJoCo, EGL</td></tr>
</tbody>
</table>

In [ ]:
# setup: the detector, hand tracker, Llama parser, dispatcher and console come
# from the pipeline (vla_pipeline). The arm is the MuJoCo MoveIt twin.
import importlib.util
import threading, time

# hide harmless warnings from other libraries
from vvla_workshop.quiet import hush_benign_warnings

hush_benign_warnings()  # silence known-benign third-party log noise

# Keep logger.info off the cell. Jupyter prints stderr in red, and that
# stream is where an info handler lands.
import logging
logging.getLogger().setLevel(logging.WARNING)

# images (OpenCV), arrays (NumPy), plots, and the buttons and text box used in section 5
import cv2, numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from vvla_workshop.ui_poll import ui_events  # jupyter_ui_poll's, with the stop button working on cells run meanwhile

# read the pipeline settings, then switch them from the real arm to the twin
from vla_pipeline.utils.config import load_config

cfg = load_config()
cfg["robot"]["backend"] = "sim"  # the MuJoCo twin instead of the Feetech bus
cfg["cameras"]["mount"]["use_ros2"] = False  # this kernel reads the webcam itself - no ROS 2 needed


## 0. A body

The twin is the MuJoCo scene under ros2_control and MoveIt. The next cell starts that bringup and connects this kernel to it with `MoveItArm`. Joint positions from here on are radians under the URDF joint names.


In [ ]:
# the MuJoCo MoveIt bringup, and this kernel's arm
from vvla_workshop.quiet import hush_benign_warnings

hush_benign_warnings()  # MoveIt INFO goes to stderr; Jupyter would show it in red
from vvla_workshop.twin_ros import TwinSession

twin = TwinSession(cfg)
arm, wrist, view = twin.arm, twin.armcam, twin.view
model = arm.robot.get_robot_model()
print(f"{model.name}: {len(arm.arm_joints)} arm joints, gripper {arm.gripper_joint}")
print(f"twin pid {twin.pid} on ROS domain {twin.domain_id}")
print(f"sim clock {arm.state_time():.3f} s")


## 1. It moves like the arm

`go_to` with a joint goal asks MoveIt for the move. MoveIt plans it and times it within each joint's `max_velocity` and `max_acceleration` in `joint_limits.yaml`, and the joint trajectory controller plays it on the twin, as it does on the arm. The samples below are the twin's joint states during a 60 degree move of the base. `go_to_rest` returns to the SRDF home pose and closes the jaws.


In [ ]:
# one planned move on the base, in radians, then back to the SRDF home pose
from sensor_msgs.msg import JointState
from vla_pipeline.robot.moveit_arm import moveit_configs

joint = "shoulder_pan_joint"
delta = 1.047  # 60 degrees
limit = moveit_configs().joint_limits["robot_description_planning"]["joint_limits"][joint]["max_velocity"]
q0 = arm.get_joints()[joint]

# record the twin's joint states, stamped on its clock, while MoveIt plans and runs the move
samples = []
sub = arm.node.create_subscription(
    JointState, "/joint_states",
    lambda m: samples.append((m.header.stamp.sec + m.header.stamp.nanosec * 1e-9, dict(zip(m.name, m.position))[joint])), 100,
)
print("move:", arm.go_to({joint: q0 + delta}))
arm.node.destroy_subscription(sub)

t, q = np.array(samples).T
t, q = t - t[0], q - q0
v = np.gradient(q, t)
print(f"peak speed {v.max():.2f} rad/s, max_velocity {limit:.2f} rad/s in joint_limits.yaml")

fig, ax = plt.subplots(figsize=(8, 3))
ax.axhline(delta, color="k", ls="--", lw=1, label="goal")
ax.plot(t, q, label="position (rad)")
ax.plot(t, v, label="speed (rad/s)")
ax.axhline(limit, color="r", ls=":", lw=1, label="max_velocity")
ax.set_xlabel("seconds on the twin's clock")
ax.legend()
plt.show()
arm.go_to_rest()


## 2. It sees like the arm

The wrist camera and the mount camera are MuJoCo camera plugins. They publish ROS images. `read()` returns `(ok, bgr)`, the same shape as a webcam. The next cell moves to the SRDF ready pose with the jaws open and shows both frames.


In [ ]:
# wrist and mount cameras from the MuJoCo plugins
arm.go_to_named("ready")
arm.go_to_named("open")
ok_w, w_img = wrist.read()
ok_v, v_img = view.read()
assert ok_w and ok_v, "a camera published no frame"
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, img, title in zip(axes, (w_img, v_img), ("wrist camera", "mount camera")):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
plt.show()


## 3. It feels like the arm

The gripper is the parallel gripper action. An empty close is asked for the SRDF close pose. The action reports whether the jaws stalled on the way there.


In [ ]:
# empty close on the gripper action, then open and park
arm.go_to_named("ready")
close = arm.named_state("close")[arm.gripper_joint]
stalled = arm.grip(close)
print(f"empty close stalled: {stalled}")
arm.go_to_named("open")
arm.go_to_rest()


## 4. Find it, reach it, grab it

To pick up an object, the arm does three things:

1. **Find it:** YOLO draws a box around it in the wrist camera image.
2. **Locate it:** the box is turned into a position on the table, in metres.
3. **Reach it:** MoveIt plans a motion that puts the open jaws around it.

The next cell turns the command into an object name and turns the arm until YOLO finds it.

In [ ]:
# the object detector, and helpers that turn a command into a COCO class name
from vla_pipeline.llm.intents import offline_parse_text
from vla_pipeline.vision.yolo_detect_npu import ObjectDetector, resolve_class

# the command to act on; try another object here
COMMAND = "pick up the ball"

# The twin's ROS participant already exists (section 0), which is what the NPU session needs.
detector = ObjectDetector(cfg)  # YOLOv26s detect; the console below reuses it
intent = offline_parse_text(COMMAND)  # the keyword stand-in for Llama
name = intent["object"]
print(f"{COMMAND!r} -> {intent} -> COCO class {resolve_class(name)!r}")
print(f"YOLO detect on {detector.provider}")

# walk the SRDF scan states with the jaws open, as the console's pick does, take a wrist-camera
# frame at each, and stop at the first that shows the object
import time
for scan in arm.scan_states():
    arm.go_to_named("open")
    arm.go_to_named(scan)
    scan_joints = arm.get_joints()
    # read() is a snapshot. The move can outlast that snapshot's freshness, so
    # wait up to the camera's one-second staleness window for a frame that
    # arrives after the arm has stopped.
    ok, frame, _stamp = wrist.frame_arrived_after(time.time(), 1.0)
    assert ok and frame is not None, "wrist camera published no frame"
    dets, cls = detector.find(frame, name)
    print(f"{scan}: " + (f"{cls}, score {dets[0].score:.2f}" if dets else "no match"))
    if dets:
        break
assert dets, f"no {name} in any scan view"
det = dets[0]

plt.figure(figsize=(6, 4.5))
plt.imshow(cv2.cvtColor(detector.draw(frame, [det]), cv2.COLOR_BGR2RGB))
plt.title(f"wrist camera at {scan}")
plt.axis("off")
plt.show()


### Finding the requested object

The arm knows its joint angles, so it knows where the camera is and which way it points. Each pixel in the image is a direction from the camera. The object sits on the table, so the line through the middle of the box meets the height of the object's centre where the object is. That gives its position from one image. `locate_detection` is the estimate the console's pick makes.

The twin knows the true position, so the plot shows how close the estimate is.

In [ ]:
# the console pick's locate: the camera's lens model, where the camera is, and the table height
from sensor_msgs.msg import CameraInfo
from vla_pipeline.behaviors.manipulation import _optical_in_base, _ros_message
from vla_pipeline.vision.yolo_detect_npu import locate_detection

pp = cfg["pick_place"]
table_z = float(pp["table_z"])
info_topic = cfg["cameras"]["arm"]["topic"].removesuffix("image_raw") + "camera_info"
info = _ros_message(arm.node, CameraInfo, info_topic)
assert info is not None, f"no camera info on {info_topic}"
optical = _optical_in_base(arm.node)  # the camera's rotation and position in base_link
# turn the pixel box into a position on the table, in metres from the arm's base
loc = locate_detection(det.box, info, optical, table_z, cls, frame, float(pp["wrist_axis_min_elongation"]))
assert loc is not None, "the box's ray does not meet the table"

# where the twin really put the object, and where the camera is
body = cls.replace(" ", "_")  # the twin's body for this class
scene = twin.scene.latest(timeout=5.0)
assert scene is not None and body in scene["bodies"], body
truth = np.array(scene["bodies"][body]["xyz"])
camera_xyz = optical[1]
error_mm = 1000 * float(np.linalg.norm(loc["xyz"][:2] - truth[:2]))
xyz = lambda v: "(" + ", ".join(f"{c:.3f}" for c in v) + ") m"
print(f"estimate: centre {xyz(loc['xyz'])}")
print(f"ground truth: centre {xyz(truth)}")
print(f"error: {error_mm:.1f} mm on the table, {1000 * abs(loc['xyz'][2] - truth[2]):.1f} mm in height")

# top view of the table: every object, the arm base, the camera, the estimate and the truth
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for other, pose in scene["bodies"].items():
    o = pose["xyz"][:2]
    ax.plot(*o, "o", color="0.8", ms=12)
    ax.annotate(other, o, xytext=(7, 7), textcoords="offset points", color="0.4")
ax.plot(0, 0, "ks", ms=10, label="arm base")
ax.plot(*camera_xyz[:2], "b^", ms=9, label="wrist camera")
ax.plot([camera_xyz[0], loc["xyz"][0]], [camera_xyz[1], loc["xyz"][1]], "b:", lw=1)
ax.plot(*truth[:2], "o", mfc="none", mec="g", ms=16, mew=2, label="true position")
ax.plot(*loc["xyz"][:2], "rx", ms=10, mew=2, label="estimate")
ax.set_aspect("equal")
ax.margins(0.15)
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("the table from above", pad=36)
# above the axes, so it cannot cover an object
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=8, frameon=False)
plt.show()

### Reaching the object

The jaws close across one side of the object. `grasp_box` turns the object's box so that side lies between the jaws, and refuses an object wider than the open jaws. The object then goes into MoveIt's planning scene, so the plan does not move the arm through it.

MoveIt Task Constructor plans the pick from where the arm is: a free move to above the object, a straight approach down, the close, and a short lift.

In [ ]:
# which side the jaws close across, and how far above the object the approach starts
from vla_pipeline.vision.yolo_detect_npu import CLASS_SIZE

box = arm.grasp_box(CLASS_SIZE[cls], loc["yaw"])
assert box is not None, f"the {cls} is too wide for the open jaws"
grip_size, yaw = box
lift = float(pp["wrist_align_pregrasp_above_m"] if cls in pp["wrist_align_classes"] else pp["pregrasp_above_m"])
arm.remember(cls, loc["xyz"], grip_size, yaw)  # the object, in MoveIt's planning scene
print(f"{cls}: the jaws close across {1000 * grip_size[0]:.0f} mm, the box is "
      + " x ".join(f"{1000 * s:.0f}" for s in grip_size) + " mm")
print(f"turned {np.degrees(yaw):.0f} deg about the vertical; the approach starts {1000 * lift:.0f} mm above it")

### Grabbing the object

The next cell runs that pick on the twin with `pick_up`, the call the console makes. The plot shows the object's height, read from the twin. If it rises with the arm, the grasp worked. The demonstration leaves the ball held so you can inspect it; Section 5 resets the twin before its interactive console starts.

In [ ]:
# the object's height, one sample per twin scene message, while the pick runs
trace = []
recording = threading.Event()
recording.set()
start = twin.scene.latest(timeout=5.0)
assert start is not None, "the twin published no scene"


def record():
    last = start["stamp"]
    while recording.is_set():
        s = twin.scene.latest(after=last, timeout=None)  # the next scene message
        last = s["stamp"]
        trace.append((last, s["bodies"][body]["xyz"][2]))


recorder = threading.Thread(target=record, daemon=True)
recorder.start()
how = arm.pick_up(cls, loc["xyz"], grip_size, lift, yaw)
# The first settled scene after ``start`` can precede the pick. Only accept a
# stopped prop state published after the arm reports the pick complete.
done = arm.state_time()
held = twin.scene.latest(after=done, settled=True, timeout=15.0)
assert how == "arrived" and held is not None, f"the pick did not hold the {cls}: {how}"
closed = arm.named_state("close")[arm.gripper_joint]
jaw = arm.get_joints()[arm.gripper_joint]
# An empty close reaches the SRDF close pose. A grip stalls short of it.
assert abs(jaw - closed) >= 0.01, (
    f"the jaws reached the empty close at {jaw:.4f} rad; they did not grip the {cls}"
)
# The plot must show a camera frame made after the raised scene state. ``read``
# can return an older frame in which the ball is still on the table.
ok_v, lifted = view.read_after(held["stamp"])
assert ok_v and lifted is not None, "the mount camera published no frame after the lift"
recording.clear()
recorder.join()

t, z = np.array(trace).T
rose_mm = 1000 * (z[-1] - z[0])
print(f"pick_up: {how}")
print(f"{cls} rose {rose_mm:.0f} mm")

# height over time, beside the mount camera after the lift
fig, (ax, ax_img) = plt.subplots(1, 2, figsize=(12, 3.8))
ax.plot(t - t[0], 1000 * (z - z[0]), label=f"{cls} height")
ax.set_xlabel("seconds on the twin's clock")
ax.set_ylabel("height above its rest (mm)")
ax.legend(loc="upper left", fontsize=8)
if ok_v:
    ax_img.imshow(cv2.cvtColor(lifted, cv2.COLOR_BGR2RGB))
ax_img.set_title("mount camera after the lift")
ax_img.axis("off")
plt.show()

print(f"Holding the {cls}. It stays in the gripper until you use the console's reset or release command.")

### What the full behavior adds

The ball remains held above the table after the cell completes. Use the **reset** button in the console below to return the arm and every object to their starting positions, or type "release" to open the jaws where they are.

When you say "pick up ..." in the console below, the arm does the same steps. It goes through the scan views until one shows the object and its pick plans, and every object it sees there goes into the planning scene. If it cannot see the object, or cannot grasp it, it tells you.

## 5. Control the digital twin

The MoveIt twin is already running. From here you control it by voice or by typing:

1. **Watch you** through your webcam.
2. **Listen to you:** Whisper turns speech into text, Llama picks a command.
3. **Take commands** in the live console.

The twin uses a private ROS 2 network that only this computer sees, so the real arm never receives its commands.


In [ ]:
# Start the interactive console from a clean scene. Section 4 deliberately
# leaves its demonstration ball held, but that must not leak into this session.
assert twin.reset_scene(), "section 5 could not reset the twin"
print("SECTION5_RESET")
# the twin from section 0 is this kernel's arm
assert importlib.util.find_spec("rclpy"), (
    "section 5 needs rclpy: source /opt/ros/jazzy/setup.bash, then restart ./run_notebooks.sh"
)
print(f"twin node pid {twin.pid} on ROS domain {twin.domain_id} (this machine only)")
print(f"arm: {type(arm).__name__}")


### Watching you

The next cell loads the pose and hand models, builds the console, and shows ten seconds of your webcam the way the two models see it. The tile titles show which processor runs each model. Without a webcam, a scripted person stands in.

In [ ]:
# your webcam, YOLO-pose and MediaPipe Hands, and the console that shows what they find
from vla_pipeline.console import TwinConsole, jpeg
from vla_pipeline.dispatch import PipelineContext, build_registry
from vla_pipeline.utils.gesture_map import mapper_from_config
from vvla_workshop.fixtures import open_webcam

cam, pose, hands = open_webcam(cfg)  # a scripted person if the webcam gives no frames
# what every behavior gets: the config, the arm, the cameras, the models and the behaviors
ctx = PipelineContext(
    cfg=cfg,
    arm=arm,
    mount_camera=cam,
    pose=pose,
    hands=hands,
    detector=detector,  # pick_place finds objects with it
    arm_camera=wrist,  # the twin's wrist camera, for grip and pick_place
    mapper=mapper_from_config(cfg),
    registry=build_registry(),
)
# the console shares the camera frames with the behaviors and draws what they do
console = TwinConsole(ctx, view=view, hint="say 'stop' to halt the arm", restore=twin.reset_scene)

# ten seconds of your webcam, as the two models see it
preview = widgets.Image(format="jpeg")
display(preview)
for _ in range(10 * 15):
    preview.value = jpeg(console.webcam_tiles())
    time.sleep(1 / 15)


### Listening to you

Your voice reaches the robot in four steps:

1. **The VAD** (voice activity detection, CPU) listens to the microphone all the time and finds where a sentence starts and ends. A sentence ends after 1.2 s of silence, so the text appears about a second after you stop talking. Sounds shorter than 0.25 s, such as a click, are dropped.
2. **Whisper** (CPU) turns that sentence into text.
3. **Llama** (iGPU) reads the text, works out what you want, and answers with one robot command, for example `pick_place` with the object `ball`.
4. The **behavior** for that command runs on the arm. A new command stops the one that is running.

Two kinds of text never reach Llama. "stop" stops the arm at once, so a stop never waits for the language model. Noise, such as a cough or keyboard clicks, is ignored. `handle_transcript` (TODO 3.2) makes these checks.

The VAD is WebRTC's voice activity detector, a small statistical model rather than a neural network; [Voice latency](../../../README.md#voice-latency) in the top-level README explains how it works. Here it runs in `SpeechSegmenter` in `vla_pipeline/audio/vad_listener.py`, with its settings under `audio:` in `config/pipeline.yaml`.

The next cell starts the microphone and asks you for one command. It prints the text Whisper heard and the command Llama made of it, but runs nothing yet. Without a microphone it uses a typed sentence; without llama-server, a keyword match stands in for Llama.

In [ ]:
# the microphone, Whisper and Llama, and the dispatcher that runs one command at a time
from vla_pipeline.audio.vad_listener import FastListener
from vla_pipeline.audio.whisper_npu import WhisperNPU
from vla_pipeline.dispatch import Dispatcher
from vla_pipeline.llm.intents import Intent, is_stop, keyword_parse
from vla_pipeline.llm.llama_intent import LlamaIntentParser

try:
    llm = LlamaIntentParser(cfg)  # Llama on the iGPU; starts llama-server if none runs
    parse = llm.parse
except Exception as e:
    llm, parse = None, keyword_parse
    print(f"llama-server unavailable ({e}) - a keyword match picks the commands")
dispatcher = Dispatcher(ctx, parse)
console.attach(dispatcher)  # the console's voice panel shows every step
listener = FastListener(cfg, WhisperNPU(cfg).transcribe)


def explain(text):
    """What the dispatcher would do with ``text``, without running it."""
    if dispatcher.is_noise(text):
        print("That is noise, so it is ignored. Say a command.")
        return
    if is_stop(text):
        print("That is a stop: it acts at once, without asking Llama.")
        return
    t0 = time.perf_counter()
    cmd = parse(text)
    ms = 1000 * (time.perf_counter() - t0)
    who = "Llama" if llm is not None else "The keyword match"
    if cmd.intent is Intent.UNKNOWN:
        print(f"{who} found no command in it ({ms:.0f} ms).")
        return
    params = ", ".join(f"{k}: {v}" for k, v in (cmd.params or {}).items() if v)
    print(f"{who} picked: {cmd.intent.value}" + (f" ({params})" if params else "") + f", in {ms:.0f} ms")


# say one command: what Whisper heard, and the command Llama picks (nothing runs yet)
print('Say a command, for example "pick up the ball". Listening for 20 s...')
heard = listener.listen_once(timeout_s=20.0, on_drop=lambda reason: print(f"(not heard: {reason})"))
mic = listener.error is None
if not mic:
    console.voice = "typed"
    example = "could you pick up the ball for me"
    print(f"no microphone ({listener.error}) - type commands under the console")
    print(f'Without a microphone, here is a typed sentence: "{example}"')
    explain(example)
elif heard is None:
    print("Nothing heard in 20 s. Is the microphone muted?")
else:
    print(f'Whisper heard: "{heard}"')
    explain(heard)


### Controlling the digital twin

Run the next cell, then speak or type. Under the console, **stop** ends the cell and parks the arm, **reset** parks the arm and puts the objects back, and **end session** also ends the cell (run the cell again to resume). Saying "stop" only halts the behavior, so a later command in the same session still works. The notebook runs one cell at a time, so click **stop** or **end session** (or the kernel's stop button) before you run the clean-up cell.

| Say | What happens |
|---|---|
| "copy my movements" | The arm follows your right hand: where it is in the image, how close it is to the camera, its tilt, and your pinch for the gripper. |
| "pick up the ball" (or bottle, book, toothbrush) | The wrist camera finds the object; the arm grasps it, lifts it and holds it out. |
| "hold this" | The jaws close on what is between them and keep holding. If you then say "copy my movements", your pinch controls the gripper. |
| "release", "drop it" | The jaws open. |
| "place it", "set it down" | After a pickup, the arm lowers the object onto the table under the jaws, opens them, and returns home. |
| "wave", "dance" | The arm waves, or dances to a beat. |
| "go home" | The arm returns to its rest pose. |
| "stop" | Everything stops at once. |

All commands: [Voice commands → behaviors](../../../README.md#voice-commands--behaviors).

In [ ]:
# the console image with its buttons and text box, redrawn FPS times a second
FPS = 15
# Start with a frame, not empty: JupyterLab may not re-measure an image that grows later.
screen = widgets.Image(value=jpeg(console.compose()), format="jpeg", layout=widgets.Layout(width="1200px"))
stop = widgets.Button(description="stop", button_style="danger")
reset = widgets.Button(description="reset", button_style="warning")
typed = widgets.Text(placeholder="or type a command, then Enter", continuous_update=False,
                     layout=widgets.Layout(width="400px"))
end = widgets.Button(description="end session")
finished = threading.Event()  # set by stop or end session


def submit(change):
    """Enter in the text box sends the typed command, the same way as a spoken one."""
    text = change["new"].strip()
    if text:
        typed.value = ""
        dispatcher.on_transcript(text)


def end_session(_=None):
    """Ends the loop below and the running behavior."""
    finished.set()
    ctx.stop_event.set()
    dispatcher.stop()  # wakes the dispatch thread, which then sees finished


stop.on_click(end_session)  # the red button ends the cell; saying "stop" only halts the behavior
reset.on_click(lambda _: console.reset_scene())
typed.observe(submit, names="value")
end.on_click(end_session)
display(widgets.VBox([screen, widgets.HBox([stop, reset, typed, end])]))

# run commands on their own thread, one at a time, listen on another, and redraw the console until stop or end session
ctx.say("Ready. Try: 'copy my movements', 'hold this', 'pick up the ball', 'pick up the bottle', "
        "'pick up the book', 'pick up the toothbrush', 'wave', 'dance', 'go home' - or 'stop'.")
dispatcher.discard()  # what was typed or said before this session does not run
worker = threading.Thread(target=dispatcher.run, args=(finished,), daemon=True, name="dispatch")
worker.start()
if mic:
    listener.start(dispatcher.on_transcript, on_level=console.on_level, on_busy=console.note_asr,
                   on_drop=lambda reason: ctx.emit("ignored", text="", reason=reason))
try:
    # running this cell again while it runs (JupyterLab's run and advance) also ends the session
    with ui_events(on_rerun=end_session) as poll:
        while not finished.is_set():
            t0 = time.perf_counter()
            poll(10)  # deliver button clicks and typed commands while this cell runs
            screen.value = jpeg(console.compose())
            time.sleep(max(0.0, 1.0 / FPS - (time.perf_counter() - t0)))
finally:
    # stop, end session, or the kernel's stop button: stop the listener and the dispatcher, wait for the last behavior, park the arm
    end_session()
    listener.stop()
    worker.join(timeout=10)
    arm.go_to_rest()
    print("session ended, arm parked")

### Running it outside the notebook

This section ran the pipeline itself (`vla_pipeline/`), which holds the finished code of every TODO. Your versions are in `workshop/project/`:

- `models/`: `npu.py` (TODOs 1.1-1.3), `hands.py` (1.4), `llm.py` (1.5-1.6)
- `ros/`: `camera_client.py` (2.3), `intent_node.py` (2.10-2.11)
- `integration/`: `dispatch.py` (3.1-3.2), `mimic.py` (3.3), `behaviors.py` (3.4)

`python -m vvla_workshop.selftest` checks all of them; `--reference` runs the same checks on the pipeline's code. To drive the twin from a terminal, with the same console in a desktop window, run `./run_pipeline.sh --digital-twin`.


## Clean up

Click **stop** or **end session** under the console (or the kernel's stop button) before you run this cell. It closes the cameras, the twin program and, if this notebook started it, Llama; the microphone closed with the console session.

In [ ]:
# the console cell ended its session; this also ends a dispatch thread it left running
finished.set()
ctx.stop_event.set()
dispatcher.stop()
worker.join(timeout=5)
# close everything that holds a device or a process
if llm is not None:
    llm.close()  # llama-server, if this notebook started it
console.close()  # stops sharing the camera frames and closes the cameras and the 3D view
hands.close()
twin.close()  # the arm client, the scene, and the twin program
print("twin closed")
